# التجربة الثانية: متى تفيد معالجة القيم المتطرفة والضوضاء؟ ولماذا تضر غالباً؟

**ضع في المجلد نفسه:** هذا الدفتر + `overlap_experiment.py` + `experiment_v2.py` + `analyze_v2.py`.

اقرأ `PROTOCOL.md` أولاً: فيه الفرضيات ومعايير الحكم، مثبّتة قبل رؤية النتائج.

قبل التشغيل: الشاحن موصول، وضع Performance/Turbo، والسكون معطّل.

In [ ]:
import os, sys, argparse, importlib
from pathlib import Path
for f in ["overlap_experiment.py", "experiment_v2.py", "analyze_v2.py"]:
    assert Path(f).exists(), f"ضع {f} في نفس مجلد الدفتر"
sys.path.insert(0, os.getcwd())
import overlap_experiment, experiment_v2 as v2
importlib.reload(overlap_experiment); importlib.reload(v2)
from IPython.display import Markdown, Image, display
print("Logical CPUs:", os.cpu_count())

def args_v2(source="openml", max_datasets=20, seeds=3, n_jobs=10,
            fracs=(0.01, 0.02, 0.05, 0.10), label_noise=(0.05, 0.10, 0.20),
            feature_outliers=(0.05, 0.10), max_rows=3000, max_features=200, out="results_v2"):
    return argparse.Namespace(source=source, max_datasets=max_datasets, seeds=seeds, n_jobs=n_jobs,
                              fracs=list(fracs), label_noise=list(label_noise),
                              feature_outliers=list(feature_outliers), max_rows=max_rows,
                              max_features=max_features, out=out)

def show_v2(out):
    display(Markdown(Path(out, "summary_v2.md").read_text(encoding="utf-8")))
    for f in ["fig_v2_detection.png", "fig_v2_gain_vs_label.png", "fig_v2_gain_vs_feature.png"]:
        if Path(out, f).exists():
            display(Image(filename=str(Path(out, f)), width=650))

## ١. اختبار سريع دون إنترنت (دقيقة إلى دقيقتين)

In [ ]:
v2.run(args_v2(source="sklearn", max_datasets=2, seeds=2, n_jobs=4, fracs=(0.02, 0.05),
               label_noise=(0.10,), feature_outliers=(0.10,), out="v2_smoke"))
%run analyze_v2.py v2_smoke
show_v2("v2_smoke")

## ٢. التجربة الأولية: 20 مجموعة × 3 بذور × 6 ظروف
**الوقت المتوقع على جهازك: نحو 1.5 إلى 3 ساعات.** يُفضَّل تشغيلها ليلاً.

للإيقاف اضغط ■، وللاستئناف أعد تشغيل الخلية نفسها.

In [ ]:
v2.run(args_v2(source="openml", max_datasets=20, seeds=3, n_jobs=10, out="results_v2"))

## ٣. التحليل

In [ ]:
%run analyze_v2.py results_v2
show_v2("results_v2")

## ٤. التجربة الكاملة (72 × 5): لا تشغّلها إلا بعد مراجعة النتائج الأولية

In [ ]:
# v2.run(args_v2(source="openml", max_datasets=72, seeds=5, n_jobs=10, out="results_v2_full"))
# %run analyze_v2.py results_v2_full